In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE CATALOG IF NOT EXISTS company_ro")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.gold")

In [0]:
company_financial_summary = spark.table("company_ro.gold.company_financial_summary")

display(company_financial_summary.limit(20))
print(company_financial_summary.columns)

In [0]:
%sql
CREATE OR REPLACE MATERIALIZED VIEW company_ro.gold.location_caen_year_stats
AS
SELECT
  an,
  judet,
  localitate,
  grupa_caen,
  cod_caen_mfinante,
  denumire_caen,
  caen_display_name,
  numar_companii,
  total_cifra_afaceri,
  total_venituri,
  total_cheltuieli,
  total_profit_net,
  total_pierdere_neta,
  total_salariati,
  total_datorii,
  total_capitaluri_proprii,
  avg_cifra_afaceri,
  avg_profit_net,
  avg_salariati
FROM (
  SELECT
    an,
    judet,
    localitate,
    grupa_caen,
    cod_caen_mfinante,
    denumire_caen,
    caen_display_name,
    COUNT(DISTINCT cui) AS numar_companii,
    SUM(cifra_afaceri) AS total_cifra_afaceri,
    SUM(venituri_totale) AS total_venituri,
    SUM(cheltuieli_totale) AS total_cheltuieli,
    SUM(profit_net) AS total_profit_net,
    SUM(pierdere_neta) AS total_pierdere_neta,
    SUM(nr_mediu_salariati) AS total_salariati,
    SUM(datorii) AS total_datorii,
    SUM(capitaluri_proprii) AS total_capitaluri_proprii,
    AVG(cifra_afaceri) AS avg_cifra_afaceri,
    AVG(profit_net) AS avg_profit_net,
    AVG(nr_mediu_salariati) AS avg_salariati
  FROM company_ro.gold.company_financial_summary
  GROUP BY
    an,
    judet,
    localitate,
    grupa_caen,
    cod_caen_mfinante,
    denumire_caen,
    caen_display_name
)

In [0]:
%sql
-- To manually refresh the materialized view:
-- REFRESH MATERIALIZED VIEW company_ro.gold.location_caen_year_stats

In [0]:
display(spark.sql("""
SELECT
  an,
  judet,
  localitate,
  caen_display_name,

  numar_companii,
  total_cifra_afaceri,
  total_profit_net,
  total_salariati
FROM company_ro.gold.location_caen_year_stats
WHERE caen_display_name IS NOT NULL
ORDER BY an DESC, total_cifra_afaceri DESC
LIMIT 100
"""))

In [0]:
display(spark.sql("""
WITH normalized AS (
  SELECT
    *,
    UPPER(
      TRANSLATE(
        judet,
        'ĂÂÎȘŞȚŢăâîșşțţ',
        'AAISSTTaaisstt'
      )
    ) AS judet_norm
  FROM company_ro.gold.location_caen_year_stats
)

SELECT
  an,
  judet,
  localitate,
  caen_display_name,

  numar_companii,
  total_cifra_afaceri,
  total_profit_net,
  total_pierdere_neta,
  total_salariati
FROM normalized
WHERE judet_norm = 'IASI'
  AND SUBSTRING(grupa_caen, 1, 2) IN ('62', '63')
ORDER BY an DESC, total_cifra_afaceri DESC
"""))